# Inspect samples

A tour of the sample contract both drivers write: `SCI`/`IVAR`/`MASK`
(+`LAYERS`) with WCS and per-position PSF metadata in the header. Run
`scripts/fetch_galaxies.py` (and optionally `scripts/fetch_backgrounds.py`)
first; this notebook only reads what they produced.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))

root = REPO / "data" / "sga"
galaxies = root / "samples" / "galaxies"
backgrounds = root / "samples" / "backgrounds"
print("data root:", root)

In [ ]:
def show(ax, image, sigma=None):
    """Asinh stretch scaled to the frame's own noise; grayscale, origin low."""
    if sigma is None:
        sigma = 1.4826 * np.median(np.abs(image - np.median(image))) or 1e-3
    ax.imshow(np.arcsinh(image / (3.0 * sigma)), origin="lower",
              cmap="gray", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])


def load_manifest(path):
    import csv
    with open(path) as f:
        return [row for row in csv.DictReader(f)]


manifest = [r for r in load_manifest(galaxies / "manifest.csv")
            if r["status"] == "written"]
print(len(manifest), "galaxy samples on disk")

## One sample, all extensions

`SCI` is (3, H, W) grz in nanomaggies — no rescaling, no clipping, negative
sky pixels intact. `IVAR` is the per-pixel inverse variance, `MASK` the DR9
MASKBITS plane, `LAYERS` the four independent mask layers (bit 0 invalid,
bit 1 bright, bit 2 source, bit 3 galaxy).

In [ ]:
row = max(manifest, key=lambda r: float(r["galaxy_frac"]))
hdul = fits.open(galaxies / row["file"])
hdul.info()
h = hdul["SCI"].header
{k: h[k] for k in ("BRICK", "SGA_ID", "PIXSCALE", "PSF_G", "PSF_R", "PSF_Z",
                   "VALIDFRC", "BUNIT")}

In [ ]:
sci, ivar = hdul["SCI"].data, hdul["IVAR"].data
mask, layers = hdul["MASK"].data, hdul["LAYERS"].data

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k, band in enumerate("grz"):
    show(axes[0, k], sci[k])
    axes[0, k].set_title(f"SCI {band}  (PSF {h[f'PSF_{band.upper()}']:.2f}\")")
axes[0, 3].imshow(ivar[1], origin="lower", cmap="gray")
axes[0, 3].set_title("IVAR r"); axes[0, 3].set_xticks([]); axes[0, 3].set_yticks([])
axes[1, 0].imshow(mask != 0, origin="lower", cmap="gray")
axes[1, 0].set_title("MASK (any bit)")
for k, (bit, name) in enumerate([(2, "BRIGHT"), (4, "SOURCE"), (8, "GALAXY")], 1):
    axes[1, k].imshow((layers & bit) != 0, origin="lower", cmap="gray")
    axes[1, k].set_title(f"LAYERS {name}")
for ax in axes[1]:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"{row['galaxy']}  (SGA {row['sga_id']}, brick {row['brick']})")
fig.tight_layout()

## A gallery across the RA range

In [ ]:
rng = np.random.default_rng(0)
picks = rng.choice(len(manifest), min(16, len(manifest)), replace=False)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, i in zip(axes.ravel(), picks):
    with fits.open(galaxies / manifest[i]["file"]) as sample:
        show(ax, sample["SCI"].data[1])
    ax.set_title(manifest[i]["galaxy"], fontsize=8)
fig.tight_layout()

## PSF metadata: the seeing spread

The consumer's forward model reads the kernel width per frame from
`PSF_G/R/Z`. The width of this distribution is what real seeing variation
contributes to identifiability, so it is worth measuring.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for band, color in zip("grz", ("C0", "C2", "C3")):
    values = np.array([float(r[f"psf_{band}"]) for r in manifest])
    ax.hist(values, bins=40, histtype="step", color=color, label=band)
    print(f"{band}: median {np.median(values):.2f}\", "
          f"5-95% {np.percentile(values, 5):.2f}-{np.percentile(values, 95):.2f}\"")
ax.set_xlabel("PSF FWHM (arcsec)"); ax.set_ylabel("samples"); ax.legend(title="band")
ax.set_title("Seeing across the sample")

## Photometric sanity

Two checks. Negative sky pixels must survive (no clipping), and
`SCI * sqrt(IVAR)` in empty pixels should be a unit Gaussian if the
inverse-variance plane calibrates the noise correctly.

In [ ]:
quiet = min(manifest, key=lambda r: float(r["galaxy_frac"]))
with fits.open(galaxies / quiet["file"]) as sample:
    q_sci = sample["SCI"].data
    q_ivar = sample["IVAR"].data
    q_layers = sample["LAYERS"].data
empty = (q_layers == 0) & np.all(q_ivar > 0, axis=0)
pulls = (q_sci[1] * np.sqrt(q_ivar[1]))[empty]
print(f"negative pixels in SCI: {(sci < 0).mean():.1%}")
print(f"pull distribution in empty pixels of {quiet['galaxy']}: "
      f"mean {pulls.mean():.3f}, std {pulls.std():.3f} (unit Gaussian expected)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pulls, bins=100, density=True, histtype="step", color="C0",
        label="SCI·√IVAR (empty px)")
grid = np.linspace(-5, 5, 200)
ax.plot(grid, np.exp(-grid**2 / 2) / np.sqrt(2 * np.pi), color="C3",
        label="unit Gaussian")
ax.set_yscale("log"); ax.set_ylim(1e-5, 1); ax.legend()
ax.set_xlabel("pull"); ax.set_title("IVAR calibrates the pixel noise")

## Volume

In [ ]:
sizes = np.array([(galaxies / r["file"]).stat().st_size for r in manifest])
print(f"{len(sizes)} samples, {sizes.sum() / 1e9:.2f} GB "
      f"(mean {sizes.mean() / 1e6:.1f} MB)")